In [1]:
# DAY 6 — MISSION CONTROL ENVIRONMENT
%pip -q install skyfield sgp4 numpy pandas requests matplotlib
%pip -q install "python-socketio[asyncio_client]" aiohttp
%pip -q install influxdb-client

print("Mission Control environment is ready.")
print("Orbital mechanics: Skyfield / SGP4")
print("Data processing: NumPy / Pandas")
print("Realtime transport: Socket.IO / aiohttp")
print("Time-series storage client: InfluxDB")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 371.1/371.1 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.1/218.1 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.5/82.5 kB 8.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 746.3/746.3 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 257.3/257.3 kB 17.9 MB/s eta 0:00:00
Mission Control environment is ready.
Orbital mechanics: Skyfield / SGP4
Data processing: NumPy / Pandas
Realtime transport: Socket.IO / aiohttp
Time-series storage client: InfluxDB


In [2]:
import requests
import numpy as np
import pandas as pd
from datetime import datetime, timezone, timedelta
from skyfield.api import EarthSatellite, load

ts = load.timescale()
now = datetime.now(timezone.utc)

# CelesTrak uydu grupları
GROUPS = {
    "stations": "Space Stations",
    "visual": "Visual Satellites",
    "weather": "Weather Satellites",
    "science": "Science Satellites",
}

def download_tle_group(group):
    url = (
        "https://celestrak.org/NORAD/elements/gp.php"
        f"?GROUP={group}&FORMAT=tle"
    )

    response = requests.get(url, timeout=30)
    response.raise_for_status()

    lines = [
        line.strip()
        for line in response.text.splitlines()
        if line.strip()
    ]

    satellites = []
    i = 0

    while i + 2 < len(lines):
        name, line1, line2 = lines[i:i+3]

        if line1.startswith("1 ") and line2.startswith("2 "):
            try:
                sat = EarthSatellite(line1, line2, name, ts)
                sat.group_name = GROUPS.get(group, group)
                satellites.append(sat)
                i += 3
                continue
            except Exception:
                pass

        i += 1

    return satellites

satellites = []
group_counts = {}

for group in GROUPS:
    try:
        items = download_tle_group(group)
        satellites.extend(items)
        group_counts[group] = len(items)
        print(f"{group:10s}: {len(items):4d} objects")
    except Exception as e:
        group_counts[group] = 0
        print(f"{group:10s}: download failed — {e}")

# Aynı NORAD katalog numarası birden fazla grupta bulunabilir.
unique = {}

for sat in satellites:
    unique[sat.model.satnum] = sat

satellites = list(unique.values())

print("\nUnique satellites:", len(satellites))
print("Data timestamp UTC:", now.isoformat())

if not satellites:
    raise RuntimeError(
        "No TLE data downloaded. Check internet access and rerun this cell."
    )

stations  :   18 objects
visual    :  155 objects
weather   :   72 objects
science   :   46 objects

Unique satellites: 285
Data timestamp UTC: 2026-10-14T11:42:35.146911+00:00


In [3]:
from skyfield.framelib import itrs
import json
import math

EARTH_RADIUS_KM = 6371.0
MAX_TRACKED = 160
ORBIT_SAMPLES = 48
ORBIT_MINUTES = 100

# Çok büyük gruplar indirilmişse görselleştirme yükünü sınırlıyoruz.
tracked_satellites = satellites[:MAX_TRACKED]
satellite_data = []

sample_minutes = np.linspace(
    0, ORBIT_MINUTES, ORBIT_SAMPLES
)

for index, sat in enumerate(tracked_satellites):
    try:
        t0 = ts.from_datetime(now)
        geocentric = sat.at(t0)

        subpoint = sat.at(t0)
        geo = sat.at(t0)

        # Dünya merkezli inertial konum/hız
        pos_gcrs = geo.position.km
        vel = geo.velocity.km_per_s

        # Dünya ile birlikte dönen ITRS koordinatları
        pos_ecef = geo.frame_xyz(itrs).km

        # Dünya üzerindeki alt nokta
        sp = sat.at(t0)
        from skyfield.api import wgs84
        ground = wgs84.subpoint_of(sp)

        orbit = []

        # Gelecek zamanlarda hesaplanan örnek yörünge
        for minute in sample_minutes:
            future_dt = now + timedelta(minutes=float(minute))
            tf = ts.from_datetime(future_dt)
            future_geo = sat.at(tf)
            ecef = future_geo.frame_xyz(itrs).km

            # Three.js: Y yukarı, Z ekseni longitude dönüşümü
            orbit.append([
                float(ecef[0] / EARTH_RADIUS_KM),
                float(ecef[2] / EARTH_RADIUS_KM),
                float(-ecef[1] / EARTH_RADIUS_KM),
            ])

        record = {
            "name": sat.name,
            "norad": int(sat.model.satnum),
            "group": getattr(sat, "group_name", "Other"),
            "lat": float(ground.latitude.degrees),
            "lon": float(ground.longitude.degrees),
            "altitude_km": float(ground.elevation.km),
            "speed_km_s": float(np.linalg.norm(vel)),
            "x": float(pos_ecef[0] / EARTH_RADIUS_KM),
            "y": float(pos_ecef[2] / EARTH_RADIUS_KM),
            "z": float(-pos_ecef[1] / EARTH_RADIUS_KM),
            "orbit": orbit,
            "source": "CelesTrak TLE / SGP4",
            "telemetry_type": "orbital_state",
        }

        satellite_data.append(record)

    except Exception as e:
        print(f"Skipped {sat.name}: {e}")

print(f"Prepared {len(satellite_data)} satellite records.")

if satellite_data:
    df_satellites = pd.DataFrame([
        {k: v for k, v in item.items() if k != "orbit"}
        for item in satellite_data
    ])

    display(
        df_satellites[
            ["name", "norad", "group", "lat", "lon",
             "altitude_km", "speed_km_s"]
        ].head(10)
    )
else:
    raise RuntimeError(
        "No satellite states prepared. Check the TLE download cell."
    )

# JSON verisi HTML dashboard'a aktarılacak.
satellite_json = json.dumps(satellite_data)

Prepared 160 satellite records.


,name,norad,group,lat,lon,altitude_km,speed_km_s
0,ISS (ZARYA),25544,Visual Satellites,44.623097,0.044554,0.0,7.658166
1,POISK,36086,Space Stations,44.623097,0.044554,0.0,7.658166
2,CSS (TIANHE),48274,Visual Satellites,37.268359,177.605821,0.0,7.683573
3,ISS (NAUKA),49044,Space Stations,44.623097,0.044554,0.0,7.658166
4,FREGAT DEB,49271,Space Stations,12.163843,173.292366,0.0,6.484450
5,CSS (WENTIAN),53239,Space Stations,37.268359,177.605821,0.0,7.683573
6,CSS (MENGTIAN),54216,Space Stations,37.268359,177.605821,0.0,7.683573
7,HRC MONOBLOCK CAMERA,66052,Space Stations,-50.963348,98.231089,0.0,7.796079
8,SZ-21 MODULE,66515,Visual Satellites,-33.126148,-53.592856,0.0,7.724822
9,DUPLEX,66906,Space Stations,15.070074,-143.620285,0.0,7.701465


In [4]:
import time
from collections import deque

event_log = deque(maxlen=100)

event_log.append({
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "level": "INFO",
    "message": f"Loaded {len(satellite_data)} orbital records",
    "source": "DATA_PIPELINE",
})

event_log.append({
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "level": "INFO",
    "message": "TLE / SGP4 propagation initialized",
    "source": "ORBIT_ENGINE",
})

event_log.append({
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "level": "NOTICE",
    "message": "Internal spacecraft telemetry unavailable from TLE",
    "source": "TELEMETRY_VALIDATION",
})

system_status = {
    "orbital_engine": "ONLINE",
    "tle_source": "ONLINE",
    "dashboard": "INITIALIZING",
    "influxdb": "NOT_CONFIGURED",
    "socketio": "NOT_STARTED",
}

print("SYSTEM STATUS")
for key, value in system_status.items():
    print(f"{key:22s} {value}")

print("\nRecent events:")
for event in event_log:
    print(
        f"[{event['level']}] {event['timestamp']} — "
        f"{event['message']}"
    )

SYSTEM STATUS
orbital_engine         ONLINE
tle_source             ONLINE
dashboard              INITIALIZING
influxdb               NOT_CONFIGURED
socketio               NOT_STARTED

Recent events:
[INFO] 2026-10-14T11:43:54.176458+00:00 — Loaded 160 orbital records
[INFO] 2026-10-14T11:43:54.176551+00:00 — TLE / SGP4 propagation initialized
[NOTICE] 2026-10-14T11:43:54.176612+00:00 — Internal spacecraft telemetry unavailable from TLE


In [6]:
import asyncio
import threading
import socketio
from aiohttp import web

# Önceden aynı hücre çalıştırıldıysa yeni sunucu başlatma.
if "mission_server_thread" not in globals():
    sio = socketio.AsyncServer(
        async_mode="aiohttp",
        cors_allowed_origins="*",
    )
    app = web.Application()
    sio.attach(app)

    @sio.event
    async def connect(sid, environ, auth=None):
        print("Dashboard connected:", sid)
        await sio.emit(
            "mission_status",
            {
                "status": "ONLINE",
                "satellite_count": len(satellite_data),
                "timestamp": datetime.now(
                    timezone.utc
                ).isoformat(),
            },
            to=sid,
        )

    @sio.event
    async def disconnect(sid, reason=None):
        print("Dashboard disconnected:", sid, reason)

    async def telemetry_publisher():
        while True:
            timestamp = datetime.now(timezone.utc).isoformat()

            # TLE kaynaklı gerçek hesaplanan alanlar
            payload = {
                "timestamp": timestamp,
                "satellite_count": len(satellite_data),
                "source": "CelesTrak TLE / SGP4",
                "records": [
                    {
                        "norad": item["norad"],
                        "name": item["name"],
                        "altitude_km": item["altitude_km"],
                        "speed_km_s": item["speed_km_s"],
                        "lat": item["lat"],
                        "lon": item["lon"],
                    }
                    for item in satellite_data[:30]
                ],
            }

            await sio.emit("telemetry_update", payload)
            await asyncio.sleep(2)

    def run_mission_server():
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)

        async def start():
            runner = web.AppRunner(app)
            await runner.setup()

            site = web.TCPSite(
                runner,
                host="0.0.0.0",
                port=8765,
            )
            await site.start()

            loop.create_task(telemetry_publisher())
            print("Socket.IO server listening on port 8765")

        loop.run_until_complete(start())
        loop.run_forever()

    mission_server_thread = threading.Thread(
        target=run_mission_server,
        daemon=True,
    )
    mission_server_thread.start()

    system_status["socketio"] = "STARTING"
else:
    print("Socket.IO server cell already started in this runtime.")

Socket.IO server listening on port 8765


In [7]:
from google.colab.output import eval_js

SOCKET_URL = eval_js(
    "google.colab.kernel.proxyPort(8765)"
)

print("Socket.IO proxy URL:")
print(SOCKET_URL)

Socket.IO proxy URL:
https://8765-m-s-kkb-use1d2-hy594fr60geu-d.us-east1-2.prod.colab.dev


In [8]:
import json
import html
from IPython.display import display, HTML

if not satellite_data:
    raise RuntimeError("Run the satellite data preparation cell first.")

SATELLITE_JSON = json.dumps(satellite_data)
SOCKET_JSON = json.dumps(
    SOCKET_URL if "SOCKET_URL" in globals() else ""
)

page = r"""
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>MISSION CONTROL</title>
<style>
*{box-sizing:border-box}
body{margin:0;background:#050912;color:#d9e6f5;
font-family:Inter,Arial,sans-serif;font-size:12px}
button,input,select{font:inherit}
#app{min-height:780px;background:
radial-gradient(circle at 52% 43%,#102039 0,#080f1c 38%,#050912 78%)}
header{height:64px;display:flex;align-items:center;justify-content:space-between;
padding:0 20px;border-bottom:1px solid #24334a;background:#080e19}
.brand{font-size:15px;letter-spacing:2px;font-weight:800;color:#f1f7ff}
.sub{font-size:10px;color:#7790ae;letter-spacing:1.5px;margin-top:5px}
.pill{padding:6px 9px;border:1px solid #29425b;border-radius:4px;
color:#7de8c1;background:#0b1b23;font-size:10px;letter-spacing:1px}
.layout{display:grid;grid-template-columns:230px minmax(300px,1fr) 270px;
gap:10px;padding:10px;min-height:700px}
.panel{background:rgba(10,19,33,.92);border:1px solid #1e3048;
border-radius:6px;overflow:hidden;min-width:0}
.panelhead{padding:13px 14px;border-bottom:1px solid #1e3048;
font-weight:700;letter-spacing:1.2px;font-size:10px;color:#a9c5e5}
.pad{padding:12px}
.muted{color:#728aa7}
input,select{width:100%;background:#07111f;color:#dbeafe;border:1px solid #243a55;
padding:9px;border-radius:4px;outline:none;margin-bottom:8px}
input:focus,select:focus{border-color:#43c8ff}
.filters{display:flex;gap:5px;flex-wrap:wrap;margin:5px 0 10px}
.filter{border:1px solid #263c56;background:#0a1525;color:#92a8c3;
padding:5px 7px;border-radius:3px;cursor:pointer}
.filter.active{border-color:#34c7f4;color:#7de8ff;background:#10283b}
#objectList{max-height:480px;overflow:auto}
.object{padding:10px;border-bottom:1px solid #17263a;cursor:pointer}
.object:hover,.object.selected{background:#10243a}
.object.selected{border-left:2px solid #43d6ff}
.objectname{font-size:11px;font-weight:700;overflow-wrap:anywhere}
.objectmeta{font-size:10px;color:#7790ae;margin-top:5px}
.center{display:flex;flex-direction:column;gap:10px;min-width:0}
#sceneWrap{position:relative;height:470px;min-height:380px;
background:#030711;border:1px solid #1e3048;border-radius:6px;overflow:hidden}
#scene{position:absolute;inset:0}
.overlay{position:absolute;z-index:2;pointer-events:none}
.sceneTitle{top:14px;left:15px;font-size:11px;letter-spacing:1.4px;
font-weight:700;text-shadow:0 1px 5px #000}
.sceneHint{bottom:12px;left:14px;color:#7d98b8;font-size:10px}
.sceneStat{top:13px;right:14px;text-align:right;color:#72e5c0;font-size:10px}
.sceneControls{position:absolute;z-index:3;right:12px;bottom:10px;display:flex;gap:5px}
.sceneControls button{background:#0a1727;color:#b8d7f2;border:1px solid #2a405b;
padding:6px 8px;border-radius:3px;cursor:pointer}
.metrics{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:8px}
.metric{background:#0a1423;border:1px solid #1e3048;border-radius:5px;padding:11px}
.metric label{font-size:9px;color:#829bb9;letter-spacing:1px}
.metric strong{display:block;font-size:18px;margin-top:8px;color:#e4f3ff}
.metric small{font-size:9px;color:#7be6c3}
.lower{display:grid;grid-template-columns:1.15fr 1fr;gap:10px}
.chart{width:100%;height:125px;display:block}
.statusrow{display:flex;justify-content:space-between;padding:8px 0;
border-bottom:1px solid #18263a;color:#8da4c0}
.ok{color:#70e5bd}.warn{color:#f6c86b}.bad{color:#ff7787}
.event{padding:8px 0;border-bottom:1px solid #19273a;line-height:1.5}
.event time{display:block;color:#647e9e;font-size:9px;margin-bottom:3px}
.rightstack{display:flex;flex-direction:column;gap:10px}
.kv{display:grid;grid-template-columns:1fr auto;gap:10px;padding:8px 0;
border-bottom:1px solid #18263a}
.kv span:first-child{color:#8199b7}
.kv span:last-child{color:#e2efff;text-align:right}
.badge{display:inline-block;font-size:9px;border:1px solid #34506c;
padding:3px 5px;border-radius:3px;color:#8ccaff}
.footer{padding:10px 18px;border-top:1px solid #1e3048;color:#647e9e;
font-size:10px;display:flex;justify-content:space-between;gap:8px;flex-wrap:wrap}
@media(max-width:1050px){
.layout{grid-template-columns:205px minmax(0,1fr)}
.rightstack{grid-column:1/-1;display:grid;grid-template-columns:1fr 1fr}
}
@media(max-width:700px){
.layout{grid-template-columns:1fr}
.rightstack{grid-column:auto;grid-template-columns:1fr}
#sceneWrap{height:390px}.metrics{grid-template-columns:repeat(2,minmax(0,1fr))}
.lower{grid-template-columns:1fr}
}
</style>
</head>
<body>
<div id="app">
<header>
  <div>
    <div class="brand">ORBITAL // MISSION CONTROL</div>
    <div class="sub">SPACE SITUATIONAL AWARENESS SYSTEM · PROTOTYPE</div>
  </div>
  <div style="text-align:right">
    <div class="pill" id="connection">● CONNECTING</div>
    <div class="sub" id="clock">UTC --:--:--</div>
  </div>
</header>

<div class="layout">
  <section class="panel">
    <div class="panelhead">01 / OBJECT EXPLORER</div>
    <div class="pad">
      <input id="search" placeholder="Search name or NORAD ID...">
      <select id="groupSelect">
        <option value="ALL">All object groups</option>
      </select>
      <div class="filters">
        <button class="filter active" data-mode="ALL">ALL</button>
        <button class="filter" data-mode="STATIONS">STATIONS</button>
        <button class="filter" data-mode="SCIENCE">SCIENCE</button>
      </div>
      <div class="muted" id="objectCount" style="margin:8px 0 10px">Loading objects...</div>
      <div id="objectList"></div>
    </div>
  </section>

  <main class="center">
    <div id="sceneWrap">
      <div class="overlay sceneTitle">EARTH ORBITAL VIEW / 3D</div>
      <div class="overlay sceneStat" id="sceneStatus">INITIALIZING RENDERER</div>
      <div id="scene"></div>
      <div class="overlay sceneHint">DRAG: ROTATE · SCROLL: ZOOM · SELECT OBJECT FROM LEFT PANEL</div>
      <div class="sceneControls">
        <button id="resetView">RESET VIEW</button>
        <button id="toggleOrbits">ORBITS: ON</button>
      </div>
    </div>

    <div class="metrics">
      <div class="metric"><label>TRACKED OBJECTS</label><strong id="mCount">--</strong><small>CATALOG SNAPSHOT</small></div>
      <div class="metric"><label>ALTITUDE</label><strong id="mAlt">--</strong><small>KM ABOVE EARTH</small></div>
      <div class="metric"><label>ORBITAL SPEED</label><strong id="mSpeed">--</strong><small>KM / S</small></div>
      <div class="metric"><label>DATA PIPELINE</label><strong id="mPipe" style="font-size:14px">READY</strong><small id="mSource">TLE / SGP4</small></div>
    </div>

    <div class="lower">
      <section class="panel">
        <div class="panelhead">02 / ORBITAL TELEMETRY</div>
        <div class="pad">
          <div class="muted" id="chartLabel">Selected object's orbital history</div>
          <canvas class="chart" id="chart"></canvas>
          <div class="kv"><span>Latitude</span><span id="lat">--</span></div>
          <div class="kv"><span>Longitude</span><span id="lon">--</span></div>
          <div class="kv"><span>Altitude source</span><span>SGP4 / TLE</span></div>
        </div>
      </section>
      <section class="panel">
        <div class="panelhead">03 / SYSTEM DIAGNOSTICS</div>
        <div class="pad">
          <div class="statusrow"><span>Orbital engine</span><b class="ok">● ONLINE</b></div>
          <div class="statusrow"><span>TLE data source</span><b class="ok">● LOADED</b></div>
          <div class="statusrow"><span>Socket.IO stream</span><b id="socketStatus" class="warn">● WAITING</b></div>
          <div class="statusrow"><span>InfluxDB storage</span><b id="dbStatus" class="warn">● OPTIONAL</b></div>
          <div class="statusrow"><span>Internal spacecraft telemetry</span><b class="warn">● NOT IN TLE</b></div>
        </div>
      </section>
    </div>
  </main>

  <aside class="rightstack">
    <section class="panel">
      <div class="panelhead">04 / SELECTED OBJECT</div>
      <div class="pad">
        <div id="selectedName" style="font-size:14px;font-weight:800;overflow-wrap:anywhere">No object selected</div>
        <div class="muted" id="selectedMeta" style="margin:6px 0 12px">NORAD --</div>
        <div class="kv"><span>Object group</span><span id="objGroup">--</span></div>
        <div class="kv"><span>Latitude</span><span id="objLat">--</span></div>
        <div class="kv"><span>Longitude</span><span id="objLon">--</span></div>
        <div class="kv"><span>Altitude</span><span id="objAlt">--</span></div>
        <div class="kv"><span>Speed</span><span id="objSpeed">--</span></div>
        <div class="kv"><span>Data type</span><span class="badge">ORBITAL STATE</span></div>
      </div>
    </section>

    <section class="panel">
      <div class="panelhead">05 / SPACE WEATHER</div>
      <div class="pad">
        <div class="statusrow"><span>Solar wind</span><b class="warn">SOURCE NEEDED</b></div>
        <div class="statusrow"><span>Kp index</span><b class="warn">NOT CONNECTED</b></div>
        <div class="statusrow"><span>Geomagnetic alerts</span><b class="muted">NO LIVE FEED</b></div>
        <p class="muted" style="line-height:1.6">
          Live space-weather measurements will be connected from a dedicated
          source in a later pipeline stage.
        </p>
      </div>
    </section>

    <section class="panel">
      <div class="panelhead">06 / MISSION EVENT LOG</div>
      <div class="pad" id="events">
        <div class="event"><time>BOOT</time>Dashboard initialized.</div>
        <div class="event"><time>DATA</time>Orbital catalog loaded.</div>
        <div class="event"><time>NOTICE</time>Internal telemetry is not available from TLE.</div>
      </div>
    </section>

    <section class="panel">
      <div class="panelhead">07 / SIMULATED BUS TELEMETRY</div>
      <div class="pad">
        <div class="muted" style="line-height:1.6">
          These are demonstration values, not real spacecraft readings.
        </div>
        <div class="kv"><span>Battery</span><span id="simBattery">--</span></div>
        <div class="kv"><span>Bus temperature</span><span id="simTemp">--</span></div>
        <div class="kv"><span>Fuel estimate</span><span id="simFuel">--</span></div>
      </div>
    </section>
  </aside>
</div>

<div class="footer">
  <span>MISSION CONTROL / ENGINEERING PROTOTYPE</span>
  <span id="dataStamp">Data timestamp: --</span>
  <span>NOT FOR OPERATIONAL FLIGHT USE</span>
</div>
</div>

<script src="https://cdn.jsdelivr.net/npm/three@0.128.0/build/three.min.js"></script>
<script src="https://cdn.jsdelivr.net/npm/three@0.128.0/examples/js/controls/OrbitControls.js"></script>
<script src="https://cdn.socket.io/4.7.5/socket.io.min.js"></script>
<script>
const SATS = __SATELLITES__;
const SOCKET_URL = __SOCKET_URL__;

let selected = 0;
let activeMode = "ALL";
let showOrbits = true;
let socket = null;
let scene, camera, renderer, controls, earth;
const satMeshes = new Map();
const orbitLines = [];

const $ = id => document.getElementById(id);
const fmt = (v, n=2) => Number.isFinite(Number(v)) ? Number(v).toFixed(n) : "--";

function addEvent(level, message) {
  const box = $("events");
  const node = document.createElement("div");
  node.className = "event";
  const time = document.createElement("time");
  time.textContent = new Date().toISOString().slice(11,19) + " UTC · " + level;
  node.appendChild(time);
  node.appendChild(document.createTextNode(message));
  box.prepend(node);
  while (box.children.length > 7) box.removeChild(box.lastChild);
}

function setupFilters() {
  const groups = [...new Set(SATS.map(s => s.group || "Other"))].sort();
  groups.forEach(group => {
    const opt = document.createElement("option");
    opt.value = group; opt.textContent = group;
    $("groupSelect").appendChild(opt);
  });

  $("search").addEventListener("input", renderList);
  $("groupSelect").addEventListener("change", renderList);
  document.querySelectorAll(".filter").forEach(btn => {
    btn.addEventListener("click", () => {
      activeMode = btn.dataset.mode;
      document.querySelectorAll(".filter").forEach(b => b.classList.toggle("active", b===btn));
      renderList();
    });
  });
}

function filteredObjects() {
  const query = $("search").value.toLowerCase().trim();
  const group = $("groupSelect").value;
  return SATS.map((s,i)=>({...s,_index:i})).filter(s => {
    const matchQuery = s.name.toLowerCase().includes(query) ||
      String(s.norad).includes(query);
    const matchGroup = group==="ALL" || s.group===group;
    const matchMode = activeMode==="ALL" ||
      (activeMode==="STATIONS" && /station/i.test(s.group)) ||
      (activeMode==="SCIENCE" && /science/i.test(s.group));
    return matchQuery && matchGroup && matchMode;
  });
}

function renderList() {
  const items = filteredObjects();
  $("objectCount").textContent = `${items.length} MATCHING / ${SATS.length} LOADED`;
  const list = $("objectList");
  list.replaceChildren();

  items.slice(0,180).forEach(s => {
    const row = document.createElement("div");
    row.className = "object" + (s._index===selected ? " selected" : "");
    const name = document.createElement("div");
    name.className = "objectname";
    name.textContent = s.name;
    const meta = document.createElement("div");
    meta.className = "objectmeta";
    meta.textContent = `NORAD ${s.norad} · ${s.group || "Other"}`;
    row.append(name,meta);
    row.addEventListener("click", () => {
      selected = s._index;
      renderList();
      updateSelected();
    });
    list.appendChild(row);
  });
}

function initScene() {
  const container = $("scene");
  scene = new THREE.Scene();
  scene.background = new THREE.Color(0x030711);

  camera = new THREE.PerspectiveCamera(
    45, container.clientWidth/container.clientHeight, 0.01, 1000
  );
  camera.position.set(2.8, 1.8, 3.5);

  renderer = new THREE.WebGLRenderer({antialias:true, alpha:false});
  renderer.setPixelRatio(Math.min(window.devicePixelRatio || 1, 2));
  renderer.setSize(container.clientWidth,container.clientHeight);
  renderer.outputEncoding = THREE.sRGBEncoding;
  container.appendChild(renderer.domElement);

  controls = new THREE.OrbitControls(camera,renderer.domElement);
  controls.enableDamping = true;
  controls.minDistance = 1.25;
  controls.maxDistance = 12;

  scene.add(new THREE.AmbientLight(0x8095bd,1.4));
  const sun = new THREE.DirectionalLight(0xffffff,2.0);
  sun.position.set(4,2,5); scene.add(sun);

  const textureLoader = new THREE.TextureLoader();
  const earthTexture = textureLoader.load(
    "https://threejs.org/examples/textures/planets/earth_atmos_2048.jpg",
    () => { $("sceneStatus").textContent = "EARTH TEXTURE LOADED"; },
    undefined,
    () => { $("sceneStatus").textContent = "EARTH TEXTURE UNAVAILABLE"; }
  );

  earth = new THREE.Mesh(
    new THREE.SphereGeometry(1,64,64),
    new THREE.MeshPhongMaterial({map:earthTexture, color:0xffffff, shininess:12})
  );
  scene.add(earth);

  const atmosphere = new THREE.Mesh(
    new THREE.SphereGeometry(1.025,48,48),
    new THREE.MeshBasicMaterial({
      color:0x2b8dff,transparent:true,opacity:0.10,side:THREE.BackSide
    })
  );
  scene.add(atmosphere);

  const starGeometry = new THREE.BufferGeometry();
  const starPositions = [];
  for(let i=0;i<1800;i++){
    const r=18+Math.random()*60;
    const theta=Math.random()*Math.PI*2;
    const phi=Math.acos(2*Math.random()-1);
    starPositions.push(
      r*Math.sin(phi)*Math.cos(theta),
      r*Math.cos(phi),
      r*Math.sin(phi)*Math.sin(theta)
    );
  }
  starGeometry.setAttribute(
    "position",new THREE.Float32BufferAttribute(starPositions,3)
  );
  scene.add(new THREE.Points(
    starGeometry,
    new THREE.PointsMaterial({color:0x9bbce8,size:0.055,transparent:true,opacity:0.8})
  ));

  SATS.forEach((sat,index) => {
    const color = index%5===0 ? 0xffcc72 : 0x4ce1f2;
    const mesh = new THREE.Mesh(
      new THREE.SphereGeometry(0.018,8,8),
      new THREE.MeshBasicMaterial({color})
    );
    mesh.position.set(sat.x,sat.y,sat.z);
    scene.add(mesh);
    satMeshes.set(index,mesh);

    if(sat.orbit && sat.orbit.length>1){
      const points = sat.orbit.map(p=>new THREE.Vector3(p[0],p[1],p[2]));
      const geometry = new THREE.BufferGeometry().setFromPoints(points);
      const line = new THREE.Line(
        geometry,
        new THREE.LineBasicMaterial({
          color,transparent:true,opacity:index===selected?0.65:0.12
        })
      );
      scene.add(line);
      orbitLines.push({index,line});
    }
  });

  $("sceneStatus").textContent = "3D SCENE READY";
  window.addEventListener("resize", resizeScene);
  $("resetView").addEventListener("click",()=>{
    camera.position.set(2.8,1.8,3.5);
    controls.target.set(0,0,0); controls.update();
  });
  $("toggleOrbits").addEventListener("click",()=>{
    showOrbits=!showOrbits;
    orbitLines.forEach(o=>o.line.visible=showOrbits);
    $("toggleOrbits").textContent="ORBITS: "+(showOrbits?"ON":"OFF");
  });
}

function resizeScene(){
  if(!renderer || !camera) return;
  const c=$("scene");
  camera.aspect=c.clientWidth/c.clientHeight;
  camera.updateProjectionMatrix();
  renderer.setSize(c.clientWidth,c.clientHeight);
}

function updateSelected() {
  const s = SATS[selected];
  if(!s) return;

  $("selectedName").textContent=s.name;
  $("selectedMeta").textContent=`NORAD ${s.norad} · ${s.source || "TLE"}`;
  $("objGroup").textContent=s.group || "Other";
  $("objLat").textContent=fmt(s.lat,3)+"°";
  $("objLon").textContent=fmt(s.lon,3)+"°";
  $("objAlt").textContent=fmt(s.altitude_km,1)+" km";
  $("objSpeed").textContent=fmt(s.speed_km_s,3)+" km/s";
  $("mAlt").textContent=fmt(s.altitude_km,1);
  $("mSpeed").textContent=fmt(s.speed_km_s,3);
  $("mCount").textContent=SATS.length;
  $("lat").textContent=fmt(s.lat,4)+"°";
  $("lon").textContent=fmt(s.lon,4)+"°";
  $("chartLabel").textContent=s.name+" · orbital-state snapshot";
  $("dataStamp").textContent="Snapshot generated: "+new Date().toISOString();
  orbitLines.forEach(o=>{
    o.line.material.opacity=o.index===selected?0.85:0.10;
  });

  // Demo-only bus readings; deliberately marked as simulated.
  $("simBattery").textContent=(82+Math.random()*15).toFixed(1)+"% · SIM";
  $("simTemp").textContent=(-5+Math.random()*18).toFixed(1)+" °C · SIM";
  $("simFuel").textContent=(45+Math.random()*20).toFixed(1)+"% · SIM";
  drawChart();
}

function drawChart(){
  const canvas=$("chart");
  const rect=canvas.getBoundingClientRect();
  const dpr=window.devicePixelRatio||1;
  canvas.width=Math.max(1,rect.width*dpr);
  canvas.height=Math.max(1,rect.height*dpr);
  const ctx=canvas.getContext("2d");
  ctx.scale(dpr,dpr);
  const w=rect.width,h=rect.height;
  ctx.clearRect(0,0,w,h);
  ctx.strokeStyle="#20344e";ctx.lineWidth=1;
  for(let i=1;i<5;i++){
    ctx.beginPath();ctx.moveTo(0,h*i/5);ctx.lineTo(w,h*i/5);ctx.stroke();
  }
  ctx.beginPath();
  for(let i=0;i<70;i++){
    const x=i/69*w;
    const y=h*0.48 + Math.sin(i*0.16)*h*0.13 +
      Math.cos(i*0.047)*h*0.09 + (Math.random()-0.5)*h*0.045;
    if(i===0)ctx.moveTo(x,y);else ctx.lineTo(x,y);
  }
  ctx.strokeStyle="#45d5ee";ctx.lineWidth=1.7;ctx.stroke();
  ctx.fillStyle="#7893b2";ctx.font="9px Arial";
  ctx.fillText("DEMONSTRATION TREND · NOT HISTORICAL MEASUREMENTS",5,11);
}

function connectSocket(){
  if(!SOCKET_URL || !window.io){
    $("connection").textContent="● LOCAL SNAPSHOT";
    $("socketStatus").textContent="● NOT CONNECTED";
    $("socketStatus").className="warn";
    addEvent("NOTICE","Using local orbital snapshot; Socket.IO unavailable.");
    return;
  }

  try{
    socket=io(SOCKET_URL,{transports:["websocket","polling"],timeout:8000});
    socket.on("connect",()=>{
      $("connection").textContent="● SOCKET.IO ONLINE";
      $("socketStatus").textContent="● ONLINE";
      $("socketStatus").className="ok";
      $("mPipe").textContent="ONLINE";
      addEvent("INFO","Socket.IO connection established.");
    });
    socket.on("disconnect",()=>{
      $("connection").textContent="● RECONNECTING";
      $("socketStatus").textContent="● DISCONNECTED";
      $("socketStatus").className="warn";
      addEvent("WARN","Socket.IO connection interrupted.");
    });
    socket.on("mission_status",data=>{
      addEvent("INFO",`Backend reports ${data.satellite_count} orbital records.`);
    });
    socket.on("telemetry_update",data=>{
      $("dataStamp").textContent="Last stream packet: "+data.timestamp;
      $("mPipe").textContent="STREAM";
    });
    socket.on("connect_error",()=>{
      $("connection").textContent="● LOCAL SNAPSHOT";
      $("socketStatus").textContent="● CONNECTION FAILED";
      $("socketStatus").className="warn";
    });
  }catch(e){
    $("connection").textContent="● LOCAL SNAPSHOT";
    addEvent("WARN","Socket.IO client could not initialize.");
  }
}

function animate(){
  requestAnimationFrame(animate);
  if(controls)controls.update();
  if(earth)earth.rotation.y+=0.00015;
  renderer.render(scene,camera);
}

function start(){
  setupFilters();
  renderList();
  updateSelected();
  initScene();
  connectSocket();
  $("mCount").textContent=SATS.length;
  $("mSource").textContent="TLE / SGP4";
  animate();
}
start();

setInterval(()=>{
  $("clock").textContent="UTC "+new Date().toISOString().slice(11,19);
},500);
</script>
</body>
</html>
"""

page = page.replace("__SATELLITES__", SATELLITE_JSON)
page = page.replace("__SOCKET_URL__", SOCKET_JSON)

display(HTML(
    '<iframe '
    'style="width:100%;height:1150px;border:0;border-radius:8px;" '
    'sandbox="allow-scripts allow-same-origin" '
    'srcdoc="' + html.escape(page, quote=True) + '"></iframe>'
))

/usr/local/lib/python3.13/dist-packages/IPython/core/display.py:724: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")
